In [ ]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
!pip install seqeval
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [ ]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [ ]:
%pip install -q transformers datasets seqeval evaluate accelerate --quiet
%pip install -q nltk --quiet

In [ ]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
print(f"✅ PyTorch: {torch.__version__}")

In [ ]:
data = [
    {
        "tokens": ["John", "works", "at", "Google"],
        "pos_tags": ["NNP", "VBZ", "IN", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "B-PP", "B-NP"],
    },
    {
        "tokens": ["She", "loves", "machine", "learning"],
        "pos_tags": ["PRP", "VBZ", "NN", "NN"],
        "chunk_tags": ["B-NP", "B-VP", "I-NP", "I-NP"],
    },
    {
        "tokens": ["I", "am", "learning", "NLP"],
        "pos_tags": ["PRP", "VBP", "VBG", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "I-VP", "B-NP"],
    },
]

In [ ]:
dataset = Dataset.from_list(data)

dataset = DatasetDict({
    "train": dataset,
    "validation": dataset,
    "test": dataset
})

print(dataset)

In [ ]:
# Get unique POS and chunk labels

pos_labels = list(set(label for example in data for label in example["pos_tags"]))
chunk_labels = list(set(label for example in data for label in example["chunk_tags"]))

print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)

In [ ]:
# Create mappings

pos_label2id = {label: i for i, label in enumerate(pos_labels)}
pos_id2label = {i: label for label, i in pos_label2id.items()}

chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}

print("POS mapping:", pos_label2id)
print("Chunk mapping:", chunk_label2id)

In [ ]:
# Convert labels to numbers

def encode_labels(example):
    example["pos_tags"] = [pos_label2id[label] for label in example["pos_tags"]]
    example["chunk_tags"] = [chunk_label2id[label] for label in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)

dataset["train"][0]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [ ]:

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])  # POS tagging
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

tokenized_datasets["train"][0]

In [ ]:
# Number of POS labels

num_labels = len(pos_labels)

print("Number of labels:", num_labels)

In [ ]:
# Load BERT model for token classification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=pos_id2label,
    label2id=pos_label2id
)

model.to(device)

In [ ]:
# Handles dynamic padding

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
# Training configuration

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,   # small dataset → slightly more epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,

    logging_steps=10,

    do_train=True,
    do_eval=True
)

In [ ]:
metric = evaluate.load("seqeval")


In [ ]:
# Metrics function

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(pos_id2label[p_val])
                curr_labels.append(pos_id2label[l_val])

        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()


In [ ]:
# Evaluate model

results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

In [ ]:

# Create inference pipeline

nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

In [ ]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

In [ ]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

In [ ]:
# Clean readable output

for token in output:
    print(f"{token['word']} → {token['entity']}")

#**POS Tagging vs Chunking — Comparison
**

*POS Tagging*

*  Assigns grammatical labels to each word
*  Word-level
*   John → NNP
*   Independent


*Chunking*

*  Groups words into meaningful phrases
*  Phrase-level
*   John → B-NP
*  Depends on POS tags


# **Observations**
1. Model Learned Basic Patterns
Achieved:
Accuracy ≈ 75%
F1 Score ≈ 0.72

Good for tiny dataset

2. Overfitting Likely
Same data used for:
train
validation
test

Model is not truly evaluated

3. BERT Still Performs Decently
Even with tiny data, it learned some structure

Shows power of pretrained models.


# **Conclusion**

BERT can be successfully fine-tuned for POS tagging
Performance is limited due to:


*   small dataset
*   lack of diversity
*   CPU training

*   Chunking requires additional modeling
*   Either separate model
*   Or multi-task architecture

Final takeaway:

POS tagging is simpler and works well with limited data
Chunking is more complex and needs better data + modeling